# FitoVision — ViT-B/16 no dataset do ciclo 4

Treina, calibra e avalia o ViT-B/16 no **mesmo split** das três CNNs do ciclo 4, para a comparação de arquiteturas (Q1) do TCC-II.

**Antes de rodar**

1. No PC: `python empacota_colab.py` (em `backend/`) e envie o conteúdo de `colab_export/` para a pasta `FitoVision_Colab` do seu Google Drive.
2. Aqui: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU** (L4 ou A100; T4 também serve, só demora mais) e **RAM alta**.
3. Rode as células na ordem. Se a sessão cair, rode tudo de novo: o treino continua da última época concluída.

Guia completo: `docs/GUIA_TREINO_VIT_COLAB.md` no repositório.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, torch
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução → Alterar o tipo de ambiente de execução → GPU"
print("torch", torch.__version__, "| CPUs:", os.cpu_count())

## 2. Google Drive

Se a sua pasta no Drive tiver outro nome, mude `DRIVE` abaixo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/FitoVision_Colab'
ZIPS = ['fitovision_codigo.zip', 'fitovision_dados_test.zip',
        'fitovision_dados_val.zip', 'fitovision_dados_train.zip']

faltam = [z for z in ZIPS if not os.path.exists(f'{DRIVE}/{z}')]
assert not faltam, f'Faltam em {DRIVE}: {faltam}'
for z in ZIPS:
    print(f'{z:<32} {os.path.getsize(f"{DRIVE}/{z}") / 1e9:6.2f} GB')

## 3. Copiar e extrair

Copia cada zip para o disco local do Colab antes de extrair (ler 124 mil imagens direto do Drive deixaria o treino várias vezes mais lento). Leva de 5 a 15 minutos.

In [ ]:
import shutil, subprocess, time

WORK = '/content/fitovision/backend'
os.makedirs(WORK, exist_ok=True)

for z in ZIPS:
    feito = f'{WORK}/.extraido_{z}'
    if os.path.exists(feito):
        print(z, '— já extraído'); continue
    t0 = time.time()
    shutil.copy(f'{DRIVE}/{z}', f'/content/{z}')
    subprocess.run(['unzip', '-q', '-o', f'/content/{z}', '-d', WORK], check=True)
    os.remove(f'/content/{z}')
    open(feito, 'w').close()
    print(f'{z} — {time.time() - t0:.0f} s')

!df -h /content | tail -1

## 4. Dependências

O Colab já traz PyTorch, torchvision, scikit-learn, pandas e matplotlib. Não instale o `requirements.txt` inteiro: ele fixa a versão do PyTorch do PC e trocaria a do Colab.

In [ ]:
!pip -q install pydantic-settings

## 5. Treino + calibração + avaliação

Pesos, logs e resultados vão direto para o Drive (`FitoVision_Colab/vit/`), então nada se perde se a sessão cair. Cada linha `Epoch N | ...` é uma época concluída; `*` marca um novo melhor modelo.

Receita: 30 épocas, batch 64, AdamW, LR 5e-5 com 2 épocas de aquecimento e cosseno, weight decay 0,05, clipping 1,0, parada antecipada com paciência 10 — o resto é idêntico às CNNs.

In [ ]:
%cd /content/fitovision/backend
OUT = f'{DRIVE}/vit'
os.makedirs(OUT, exist_ok=True)

!python run_vit.py \
    --weights-dir "{OUT}/weights" --logs-dir "{OUT}/logs" --results-dir "{OUT}/results" \
    --workers {os.cpu_count()} | tee -a "{OUT}/execucao.log"

## 6. Resultados

In [ ]:
import json
import pandas as pd
from IPython.display import display

for nome in ('teste_original', 'teste_novo', 'teste_completo'):
    p = f'{OUT}/results/{nome}/metrics_comparison.csv'
    if os.path.exists(p):
        print(nome); display(pd.read_csv(p))

for p in (f'{OUT}/weights/vit_b_16_binary_calibration.json',):
    if os.path.exists(p):
        print(json.dumps(json.load(open(p)), indent=2))

h = json.load(open(f'{OUT}/logs/vit_b_16_binary_history.json'))
print(f"Treino: {h['elapsed_s'] / 3600:.1f} h | melhor F1 macro (val) {h['best_val_f1']:.4f} na época {h['best_epoch']}")
display(pd.DataFrame(h['history'])[['epoch', 'train_loss', 'val_loss', 'val_bal_acc', 'val_f1_macro', 'lr']])

## 7. Pacote para levar de volta ao PC

Gera `FitoVision_Colab/vit_resultados.zip`. Baixe pelo Drive e extraia **dentro de `backend/`**: os arquivos caem em `weights/vit/`, `results/vit/` e `logs/vit/`.

In [ ]:
import zipfile

destino = f'{DRIVE}/vit_resultados.zip'
with zipfile.ZipFile(destino, 'w', zipfile.ZIP_STORED) as z:
    for sub in ('weights', 'results', 'logs'):
        for raiz, _, nomes in os.walk(f'{OUT}/{sub}'):
            for n in nomes:
                if n.endswith('_last.pt'):
                    continue
                caminho = os.path.join(raiz, n)
                z.write(caminho, os.path.join(sub, 'vit', os.path.relpath(caminho, f'{OUT}/{sub}')))
    z.write(f'{OUT}/execucao.log', 'logs/vit/execucao.log')
print(destino, f'{os.path.getsize(destino) / 1e6:.0f} MB')

Terminou? **Ambiente de execução → Desconectar e excluir o ambiente de execução**, para não gastar unidades de computação com a GPU parada.